## Current workflow review copy

Synchronized with the completed 23 September 2026 uncalibrated engine run. Cluster assignments and household sampling are retained; thermal simulations now use cluster-median construction profiles at three median floor areas. Notebooks 05 onward use outputs/current. Notebooks 03–04 have no saved demonstration outputs in the current source. Read [the workflow](../../docs/ENGINE_WORKFLOW.md) and [interpretation](../../docs/INTERPRETATION.md). Packaging does not rerun calculations. Raw data, address-level outputs and interactive payloads are excluded.


In [ ]:
# Review copy: read saved results without running the research pipeline.
import os
if os.environ.get("ENERMAP_ENABLE_MODEL_RUN") != "1":
    raise RuntimeError("Review copy: source inputs and third-party engines are excluded. Read saved results or use the Streamlit app.")
import sys
from pathlib import Path
_repo = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "config.py").exists())
sys.path.insert(0, str(_repo))


**Retained upstream method.** The scientific preprocessing and household methods are unchanged. Authoritative demand results are now produced by notebooks 05–12 in `outputs/current`. No calibration stage is run. Raw illustrative schedule totals in this notebook are not the final annual electricity or DHW account.

# 04 · Occupancy, heating regimes and loads: the CREST/CHAP layer

UK EPCs carry no occupants, so the behavioural inputs of the UBEM come from a stochastic
generator. `uk_ubem_schedule_generator/uk_ubem_schedules.py` is a transparent
re-implementation of the logic in **McKenna et al. (2018), *A stochastic multi-energy
simulation model for UK residential buildings* (CREST Heat and Power, CHAP)** and StROBe:

* four-state occupancy per household member (away / home-active / home-asleep) from
  person templates (full-time, part-time, retired, students, children);
* a **heating regime sampled from the EFUS categories of CHAP's Table 5** (once daily all
  day, evening only, twice daily short/sustained, three periods, or "heating whenever
  actively occupied"), with a thermostat setpoint drawn from that category's mean ± SD and a
  **heating season** (start month, length) sampled from the EFUS marginals (Fig. 6);
* lighting and appliance fractions following active occupancy, people gains 131 W active /
  74 W asleep (CHAP), DHW draws of 40 l per person-day at 44.7 °C (EST).

Outside the enabled periods CHAP delivers no heat; here a 12 °C frost setback is kept, as the
generator's README explains.

What this notebook adds is the **link to the stock**: a household archetype per dwelling
(`ukubem.schedules.sample_household`, a prior on accommodation type × floor area × tenure
anchored on Census 2021 household-size marginals), the hourly arrays pyBuildingEnergy
consumes, and a check of the sampled population against the EFUS statistics.

In [ ]:
import matplotlib.pyplot as plt; plt.rcParams["savefig.dpi"] = 300   # publication resolution for every saved figure
import sys, json, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path.cwd()))
from config import *              # paths + modelling choices (edit config.py, not the notebooks)
import pbc_helpers as H
H.setup_plots()

import ukubem
from ukubem.engine import PBE, run_building, annual_summary
from ukubem.geometry import UKGeometryAssumptions, build_bui, rdsap_default_window_area
from ukubem.schedules import sample_household, dwelling_schedule, deterministic_schedule
from ukubem.systems import delivered_energy, SYSTEM_DEFAULTS
from ukubem.runner import simulate_rows, simulate_stock, aggregate_lsoa, mape
pybui = PBE.load(PBE_SRC)
UBEM_OUT = CURRENT_OUT / "method_examples"; UBEM_OUT.mkdir(exist_ok=True)
FIG_UBEM = CURRENT_OUT / "figures"; FIG_UBEM.mkdir(exist_ok=True)
print(f"pyBuildingEnergy from {PBE_SRC}   |   cores: {N_JOBS}")

In [ ]:
inputs = pd.read_parquet(EXPORT_DIR / "dwelling_inputs.parquet")
from joblib import Parallel, delayed
sample = inputs.sample(1500, random_state=SEED)

def one(did, r):
    rng = np.random.default_rng([SEED, int(did)])
    hh = sample_household(r.accommodation_type, r.floor_area_final, r.tenure, rng)
    s = dwelling_schedule(str(did), hh, r.floor_area_final, year=2021, seed=int(rng.integers(2**31)), timestep_minutes=60)
    return {"dwelling_id": did, "type": r.accommodation_type, "tenure": r.tenure, "A": r.floor_area_final, "household": hh, "people": s["people"],
            "pattern": s["heating_pattern"], "setpoint": s["setpoint_c"], "on_share": s["heating_share_of_hours"],
            "season_start": s["season_start_month"], "season_len": s["season_length_months"],
            "gains_W_m2": s["gains_w"].mean() / r.floor_area_final, "dhw_kWh": s["dhw_kWh_year"], "elec_kWh": s["elec_kWh"],
            "daily_on": s["heating_on"].reshape(365, 24).mean(axis=0), "daily_gain": s["gains_w"].reshape(365, 24).mean(axis=0)}
t0 = time.time()
S = pd.DataFrame(Parallel(n_jobs=N_JOBS)(delayed(one)(d, r) for d, r in sample.iterrows()))
print(f"{len(S):,} household-years generated in {time.time()-t0:.0f} s")

## 1 · Who lives where

In [ ]:
ct = pd.crosstab(S["type"], S["household"], normalize="index") * 100
print(f"mean household size {S['people'].mean():.2f} (Census 2021 England 2.36); by type: " + ", ".join(f"{k} {v:.2f}" for k, v in S.groupby('type')['people'].mean().items()))
fig, ax = plt.subplots(figsize=(11, 3.8)); ct.plot.bar(stacked=True, ax=ax, width=0.8, colormap="tab10", rot=0)
ax.set_ylabel("% of dwellings"); ax.set_title("sampled household archetype by accommodation type"); ax.legend(fontsize=7, ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.15))
plt.tight_layout(); fig.savefig(FIG_UBEM / "04_household_by_type.png", bbox_inches="tight"); plt.show()

## 2 · Heating regimes vs the EFUS statistics used by CHAP

In [ ]:
efus = pd.Series({"P1_ALL_DAY_FROM_WAKE": 6.3, "P2_EVENING_THROUGH_NIGHT": 4.3, "P3_EVENING_SUSTAINED": 2.3, "P5_TWO_SHORT": 9.2,
                  "P6_MORNING_SHORT_EVENING_SUSTAINED": 24.6, "P8_THREE_PERIODS": 9.4, "P9_ACTIVE_OCCUPANCY": 25.8})
efus = efus / efus.sum() * 100
got = S["pattern"].value_counts(normalize=True) * 100
tab = pd.DataFrame({"EFUS Table 5 (renormalised, %)": efus, "sampled (%)": got.reindex(efus.index)}).round(1)
tab["setpoint mean (°C)"] = S.groupby("pattern")["setpoint"].mean().reindex(efus.index).round(1)
tab["heating available, % of hours"] = (S.groupby("pattern")["on_share"].mean().reindex(efus.index) * 100).round(1)
tab["season length (months)"] = S.groupby("pattern")["season_len"].mean().reindex(efus.index).round(1)
tab.to_csv(UBEM_OUT / "04_regimes_vs_efus.csv")
tab

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 3.8))
S["season_start"].value_counts().sort_index().reindex(range(1, 13), fill_value=0).plot.bar(ax=axes[0], color="steelblue", rot=0); axes[0].set_title("heating season start month (EFUS Fig. 6: Oct 53 %, Nov 26 %)")
S["season_len"].value_counts().sort_index().reindex(range(1, 13), fill_value=0).plot.bar(ax=axes[1], color="darkorange", rot=0); axes[1].set_title("heating season length, months (EFUS: mode 5-6)")
axes[2].hist(S["setpoint"], bins=25, color="seagreen"); axes[2].set_title(f"thermostat setpoint, mean {S['setpoint'].mean():.1f} °C (EFUS 19.7-20.7 by category)")
plt.tight_layout(); fig.savefig(FIG_UBEM / "04_season_and_setpoint.png", bbox_inches="tight"); plt.show()

## 3 · Mean daily profiles by regime (the CHAP "Fig. 12" view, before the building)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.2))
for pat, grp in S.groupby("pattern"):
    axes[0].plot(range(24), np.mean(np.vstack(grp["daily_on"]), axis=0), label=f"{pat[:2]} (n={len(grp)})")
axes[0].set_xlabel("hour"); axes[0].set_ylabel("share of days with heating available"); axes[0].set_title("heating availability by EFUS regime (annual mean)"); axes[0].legend(fontsize=7)
for hh, grp in S.groupby("household"):
    axes[1].plot(range(24), np.mean(np.vstack(grp["daily_gain"]), axis=0) / grp["A"].mean(), label=hh)
axes[1].set_xlabel("hour"); axes[1].set_ylabel("W/m²"); axes[1].set_title("internal gains by household archetype (annual mean)"); axes[1].legend(fontsize=7)
plt.tight_layout(); fig.savefig(FIG_UBEM / "04_daily_profiles.png", bbox_inches="tight"); plt.show()
print(f"stock means: gains {S['gains_W_m2'].mean():.2f} W/m², DHW {S['dhw_kWh'].mean():,.0f} kWh/yr, appliances + lighting {S['elec_kWh'].mean():,.0f} kWh/yr (BREDEM levels are applied in the runs)")

## 4 · How much the regime matters for one building

The same semi-detached house run with every EFUS category (same thermostat 20 °C, season
October–April) — the behavioural spread the missing loads archetype would otherwise hide.

In [ ]:
from joblib import Parallel, delayed
A = UKGeometryAssumptions()
did = inputs[(inputs.accommodation_type == "Semi-detached") & (inputs.age_band_resolved == "D") & (inputs.wall_insulation == "Uninsulated")].index[0]
r = inputs.loc[did]
import uk_ubem_schedules as ukg
patterns = list(ukg.HEATING_PATTERNS)

def run_pattern(pat):
    s = dwelling_schedule(str(did), "working_couple", r.floor_area_final, year=2021, seed=SEED, timestep_minutes=60,
                          heating_pattern=pat, comfort_setpoint_c=20.0, season_mode="fixed_dates")
    b, d = build_bui(r, A, schedule=s)
    h, a, secs = run_building(b, EPW, str(PBE_SRC), external_internal_gains_w=s["gains_w"])
    res = annual_summary(h, a, d["A_floor"])
    return {"pattern": pat, "on_share": s["heating_share_of_hours"], "Q_H_kWh_m2": res["Q_H_kWh_m2"], "T_op_winter": res["T_op_heating_season_C"]}
det_b, det_d = build_bui(r, A); h, a, _ = run_building(det_b, EPW, str(PBE_SRC)); det_res = annual_summary(h, a, det_d["A_floor"])
R = pd.DataFrame(Parallel(n_jobs=min(N_JOBS, len(patterns)))(delayed(run_pattern)(p) for p in patterns)).set_index("pattern")
R.loc["two-period timer regime 19.5/16 °C"] = [det_res["heating_hours"] / 8760, det_res["Q_H_kWh_m2"], det_res["T_op_heating_season_C"]]
R.to_csv(UBEM_OUT / "04_regime_sensitivity.csv")
fig, ax = plt.subplots(figsize=(10, 3.8)); R["Q_H_kWh_m2"].plot.barh(ax=ax, color="steelblue"); ax.set_xlabel("space-heating need [kWh/m²/yr]")
ax.set_title(f"dwelling {did} ({r.accommodation_type}, band {r.age_band_resolved}, {r.wall_insulation.lower()} {r.wall_construction_resolved.lower()}): regime sensitivity")
plt.tight_layout(); fig.savefig(FIG_UBEM / "04_regime_sensitivity.png", bbox_inches="tight"); plt.show()
R.round(2)

**Next:** `05_archetype_run.ipynb` — the archetype tier of the UBEM (minutes, not
hours) and its first comparison with DESNZ meter data.

## 5 · CHAP microdata check

The generator now samples the EFUS microdata bundled from `CHAP_model_1.0.xlsm` (`chap_microdata.py`):
heating-regime shares, timer start / duration distributions per category, period and day type,
thermostat distributions per category, and the CREST four-state occupancy chain (ten-minute
transition matrices per household size). Category 2 is "on in the evening for all day and night",
implemented as continuous heating in season (as CHAP does), not "evening through night".

In [ ]:
from ukubem.schedules import ukg
import chap_microdata as chap
arrays, meta = chap.load()
cfg = ukg.SimulationConfig.for_year(2021, timestep_minutes=60, master_seed=SEED)
legacy = ukg.SimulationConfig.for_year(2021, timestep_minutes=60, master_seed=SEED, timing_source="assumed", setpoint_source="normal", occupancy_mode="person_templates")
rows, act_new, act_old = [], [], []
archs = list(ukg.HOUSEHOLD_ARCHETYPES)
for i in range(400):
    a = archs[i % len(archs)]; spec = ukg.DwellingSpec(f"c{i}", a, 90.0)
    r = ukg.generate_dwelling(spec, cfg); m = r.metadata; f = r.frame; prog = m.get("sampled_timer_programme") or {}
    rows.append({"pattern": m["sampled_heating_pattern"], "setpoint": m["sampled_comfort_setpoint_c"], "size": len(ukg.HOUSEHOLD_ARCHETYPES[a]),
                 "wd1_start": prog.get("WD", [(np.nan, np.nan)])[0][0], "heat_share": float((f["heating_enabled"] > 0).mean())})
    if a == "working_couple":
        loc = f.index.tz_convert(cfg.timezone); act_new.append(f["occupants_active"].groupby(loc.hour).mean().to_numpy())
        fo = ukg.generate_dwelling(spec, legacy).frame; act_old.append(fo["occupants_active"].groupby(loc.hour).mean().to_numpy())
d = pd.DataFrame(rows)
ref = {v["pattern_id"]: v["relative_share"] for v in meta["categories"].values() if v.get("pattern_id")}
fig, axes = plt.subplots(2, 2, figsize=(15, 9.5))
pd.DataFrame({"CHAP / EFUS 2011 (n = 2,142)": pd.Series(ref), "sampled (400 dwellings)": d["pattern"].value_counts(normalize=True)}).plot.bar(ax=axes[0, 0], rot=35, width=0.8)
axes[0, 0].set_ylabel("share"); axes[0, 0].set_title("heating-regime categories"); axes[0, 0].tick_params(labelsize=7)
bins = np.arange(0, 24.001, 0.25); p6 = d[d["pattern"] == "P6_MORNING_SHORT_EVENING_SUSTAINED"]
axes[0, 1].bar(bins[:-1] + 0.125, arrays["start|6|1|WD"], width=0.25, alpha=0.45, label="CHAP microdata: P6 weekday period-1 start")
axes[0, 1].bar(bins[:-1] + 0.125, np.histogram(p6["wd1_start"].dropna(), bins=bins)[0] / max(len(p6), 1), width=0.25, alpha=0.45, color="crimson", label=f"sampled (n={len(p6)})")
axes[0, 1].set_xlim(4, 12); axes[0, 1].set_xlabel("hour of day"); axes[0, 1].set_ylabel("probability per 15 min"); axes[0, 1].legend(fontsize=8); axes[0, 1].set_title("timer start times")
mix = sum(ref[p.pattern_id] * arrays[f"temp|{p.efus_category}"] for p in ukg.HEATING_PATTERNS.values()); mix = mix / mix.sum()
axes[1, 0].bar(np.arange(len(mix)), mix, width=0.9, alpha=0.45, label="CHAP microdata (share-weighted mixture)")
axes[1, 0].hist(d["setpoint"], bins=np.arange(9.5, 30.5, 1), density=True, histtype="step", lw=1.8, color="crimson", label="sampled")
axes[1, 0].set_xlim(10, 30); axes[1, 0].set_xlabel("thermostat setting [°C]"); axes[1, 0].legend(fontsize=8); axes[1, 0].set_title("thermostat settings")
axes[1, 1].plot(range(24), np.mean(act_new, axis=0), label="CREST four-state chain (CHAP matrices)", color="navy")
axes[1, 1].plot(range(24), np.mean(act_old, axis=0), label="person templates (earlier assumption)", color="grey", ls="--")
axes[1, 1].set_xlabel("hour"); axes[1, 1].set_ylabel("active occupants at home"); axes[1, 1].set_title("working couple: mean active occupancy over the year"); axes[1, 1].legend(fontsize=8)
plt.tight_layout(); fig.savefig(FIG_UBEM / "04_chap_microdata_check.png", bbox_inches="tight"); plt.show()
print("heating share of hours by category:", d.groupby("pattern")["heat_share"].mean().round(3).to_dict())
print(f"setpoint sampled mean {d['setpoint'].mean():.2f} °C (sd {d['setpoint'].std():.2f}); microdata mixture mean {(mix * np.arange(len(mix))).sum():.2f} °C")